In [ ]:
# importing libraries
import pandas as pd
from pathlib import Path
import numpy as np

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    """Find repo root by walking upward until expected project markers are found."""
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "config").exists() and (path / "notebooks").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find repo root. Launch notebook from inside the repository root.")

REPO_ROOT = find_repo_root()

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PREPROCESSING_DATA_DIR = REPO_ROOT / "data" / "processed" / "preprocessing"
OBJ2_DATA_DIR = REPO_ROOT / "data" / "processed" / "objective2_demand"
OBJ2_OUTPUT_DIR = REPO_ROOT / "outputs" / "objective2_demand"
OBJ2_MODEL_DIR = OBJ2_OUTPUT_DIR / "models"
OBJ2_VALIDATION_DIR = OBJ2_OUTPUT_DIR / "validation"
CONFIG_DIR = REPO_ROOT / "config"
LOOKUP_DIR = CONFIG_DIR / "lookups"

for path in [OBJ2_DATA_DIR, OBJ2_OUTPUT_DIR, OBJ2_MODEL_DIR, OBJ2_VALIDATION_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
# Read files

demand = pd.read_parquet(PREPROCESSING_DATA_DIR / "demand_hist_daily.parquet")
weather = pd.read_parquet(PREPROCESSING_DATA_DIR / "weather_hist_daily.parquet")
holiday = pd.read_csv(PREPROCESSING_DATA_DIR / "holiday_calendar_2010_2045.csv")


In [ ]:
# data types of demand data
demand.dtypes

In [ ]:
demand.head()

In [ ]:
demand.shape

In [ ]:
demand.isna().sum()

In [ ]:
demand.duplicated().sum()

In [ ]:
# Get the first available date in demand data
demand_min_date = demand["date"].min()

# Get the last available date in demand data
demand_max_date = demand["date"].max()

# Count how many unique dates are present
demand_unique_dates = demand["date"].nunique()

# Create the full continuous daily date range between min and max date
demand_full_range = pd.date_range(demand_min_date, demand_max_date, freq="D")

# Find dates missing from the demand dataframe
demand_missing_dates = demand_full_range.difference(demand["date"].drop_duplicates())

# Print summary
print("Demand min date:", demand_min_date)
print("Demand max date:", demand_max_date)
print("Demand unique dates:", demand_unique_dates)
print("Demand missing dates count:", len(demand_missing_dates))
print("Demand missing dates sample:", demand_missing_dates[:10])

In [ ]:
# data types of weather data
weather.dtypes

In [ ]:
weather.head()

In [ ]:
weather.shape

In [ ]:
weather.isna().sum()

In [ ]:
weather.duplicated().sum()

In [ ]:
# Get the first available date in weather data
weather_min_date = weather["date"].min()

# Get the last available date in weather data
weather_max_date = weather["date"].max()

# Count how many unique dates are present
weather_unique_dates = weather["date"].nunique()

# Create the full continuous daily date range between min and max date
weather_full_range = pd.date_range(weather_min_date, weather_max_date, freq="D")

# Find dates missing from the weather dataframe
weather_missing_dates = weather_full_range.difference(weather["date"].drop_duplicates())

# Print summary
print("Weather min date:", weather_min_date)
print("Weather max date:", weather_max_date)
print("Weather unique dates:", weather_unique_dates)
print("Weather missing dates count:", len(weather_missing_dates))
print("Weather missing dates sample:", weather_missing_dates[:10])

In [ ]:
# Count how many unique regions exist for each date
weather_region_count = weather.groupby("date")["region"].nunique()

# Show how many dates have 1 region, 2 regions, etc.
print(weather_region_count.value_counts())

# Show dates where both expected regions are not present
print(weather_region_count[weather_region_count != 2])

In [ ]:
# data types of holiday data
holiday.dtypes

In [ ]:
# Standardise date column in holiday dataframe
holiday["date"] = pd.to_datetime(holiday["date"])

In [ ]:
# data types of holiday data after changing
holiday.dtypes

In [ ]:
holiday.head()

In [ ]:
holiday.shape

In [ ]:
holiday.isna().sum()

In [ ]:
holiday.duplicated().sum()

In [ ]:
# Get the first available date in holiday data
holiday_min_date = holiday["date"].min()

# Get the last available date in holiday data
holiday_max_date = holiday["date"].max()

# Count how many unique dates are present
holiday_unique_dates = holiday["date"].nunique()

# Create the full continuous daily date range between min and max date
holiday_full_range = pd.date_range(holiday_min_date, holiday_max_date, freq="D")

# Find dates missing from the holiday dataframe
holiday_missing_dates = holiday_full_range.difference(holiday["date"].drop_duplicates())

# Print summary
print("Holiday min date:", holiday_min_date)
print("Holiday max date:", holiday_max_date)
print("Holiday unique dates:", holiday_unique_dates)
print("Holiday missing dates count:", len(holiday_missing_dates))
print("Holiday missing dates sample:", holiday_missing_dates[:10])

In [ ]:
# Reshape weather data from long format to wide format
# Current long format has one row per date + region
# We want one row per date, with separate columns for each region's weather values
weather_wide = weather.pivot(
    index="date",
    columns="region",
    values=["tasmin_c", "tasmax_c", "tmean_c"]
)

# After pivot, columns become a multi-level index like:
# ('tasmin_c', 'eng_wales'), ('tasmin_c', 'scotland'), etc.
# Flatten them into single column names
weather_wide.columns = [
    f"{var}_{region}_c".replace("_c_", "_")
    for var, region in weather_wide.columns
]

# Bring date back from index to a normal column
weather_wide = weather_wide.reset_index()

# Rename columns to match the final required naming format
weather_wide = weather_wide.rename(
    columns={
        "tasmin_c_eng_wales": "tasmin_eng_wales_c",
        "tasmax_c_eng_wales": "tasmax_eng_wales_c",
        "tmean_c_eng_wales": "tmean_eng_wales_c",
        "tasmin_c_scotland": "tasmin_scotland_c",
        "tasmax_c_scotland": "tasmax_scotland_c",
        "tmean_c_scotland": "tmean_scotland_c",
    }
)

In [ ]:
weather_wide.head(10)

In [ ]:
## Quality check
weather[
    (weather["date"] == pd.Timestamp("2010-01-01")) &
    (weather["region"] == "scotland")
]

In [ ]:
# Create calendar columns from date


# Define the required Task 1 date range
start_date = pd.Timestamp("2010-01-01")
end_date = pd.Timestamp("2024-12-31")

# Create calendar columns from date
calendar = pd.DataFrame({"date": pd.date_range(start_date, end_date, freq="D")})

# Extract year from date
calendar["year"] = calendar["date"].dt.year

# Extract month from date
calendar["month"] = calendar["date"].dt.month

# Extract day of week from date
# Monday = 0, Sunday = 6
calendar["day_of_week"] = calendar["date"].dt.dayofweek

# Mark weekend days: Saturday (5) and Sunday (6)
calendar["is_weekend"] = calendar["day_of_week"].isin([5, 6]).astype(int)


In [ ]:
# Merge all pieces into final training dataset

demand_model_training_data = (
    calendar
    .merge(demand[["date", "nd_daily_mwh"]], on="date", how="left")
    .merge(holiday, on="date", how="left")
    .merge(weather_wide, on="date", how="left")
)

In [ ]:
# Final column order

final_cols = [
    "date",
    "year",
    "month",
    "day_of_week",
    "nd_daily_mwh",
    "is_weekend",
    "is_holiday_eng_wales",
    "is_holiday_scotland",
    "is_holiday_gb_any",
    "tasmin_eng_wales_c",
    "tasmax_eng_wales_c",
    "tmean_eng_wales_c",
    "tasmin_scotland_c",
    "tasmax_scotland_c",
    "tmean_scotland_c",
]

demand_model_training_data = demand_model_training_data[final_cols].copy()

In [ ]:
demand_model_training_data

In [ ]:
# Check final row count
print("Final shape:", demand_model_training_data.shape)

# Check column order and names
print("\nFinal columns:")
print(demand_model_training_data.columns.tolist())

# Check missing values in final dataset
print("\nMissing values:")
print(demand_model_training_data.isna().sum())

# Check duplicate dates
print("\nDuplicate dates:")
print(demand_model_training_data.duplicated(subset=["date"]).sum())

# Check date range
print("\nMin date:", demand_model_training_data["date"].min())
print("Max date:", demand_model_training_data["date"].max())
print("Unique dates:", demand_model_training_data["date"].nunique())

In [ ]:
# Save final Task 1 output
demand_model_training_data.to_parquet(OBJ2_DATA_DIR / "demand_model_training_data.parquet", index=False)

print("Saved file:", OBJ2_DATA_DIR / "demand_model_training_data.parquet")


In [ ]:
# Save final Task 1 output as csv
demand_model_training_data.to_csv(OBJ2_VALIDATION_DIR / "demand_model_training_data.csv", index=False)


In [ ]:
readme_text = """
# demand_model_training_README

## Purpose
This README documents how `demand_model_training_data.parquet` was created for Objective 2 - Task 1.

This dataset is the final historic daily model input table used to train the climate-sensitive daily demand model.

Primary target: `nd_daily_mwh`
Granularity: `one row per `date`
Coverage: `2010-01-01` to `2024-12-31`
Main output file: `demand_model_training_data.parquet`
Final output shape: `5479 rows, 15 columns`

---

## Input files

### 1. `demand_hist_daily.parquet`: Historic daily demand input

Columns available:
- `date`
- `nd_daily_mwh`
- `nd_mean_mw`
- `tsd_daily_mwh`


### 2. `weather_hist_daily.parquet`: Historic daily weather input in long format

Columns:
- `date`
- `region`
- `tasmin_c`
- `tasmax_c`
- `tmean_c`

Expected regions:
- `eng_wales`
- `scotland`

### 3. `holiday_calendar_2010_2045.csv`: Holiday calendar input

Columns:
- `date`
- `is_holiday_eng_wales`
- `is_holiday_scotland`
- `is_holiday_gb_any`

---

## Initial data exploration and QA checks

### Initial dataset shapes:
- `demand_hist_daily.parquet`: 5479 rows x 4 columns
- `weather_hist_daily.parquet`: 10958 rows x 5 columns
- `holiday_calendar_2010_2045.csv`: 13149 rows x 4 columns


### Missing value results: 0 missing values in all three inputs

### Duplicate check results: 0 full-row duplicates in all three inputs

### Date coverage check results:
- demand date range: 2010-01-01 to 2024-12-31
- weather date range: 2010-01-01 to 2024-12-31
- holiday date range: 2010-01-01 to 2045-12-31


---


## Weather reshaping logic

### Why reshaping was needed
The weather input file is in long format, meaning there is one row for each `date`, `region`

Example structure before reshape:
- 2010-01-01, eng_wales, ...
- 2010-01-01, scotland, ...
- 2010-01-02, eng_wales, ...
- 2010-01-02, scotland, ...

However, the final Task 1 modelling dataset needs: one row per `date`

### Reshaping method used
A pivot was applied using:
- `index = "date"`
- `columns = "region"`
- `values = ["tasmin_c", "tasmax_c", "tmean_c"]`

This produced a wide table where each date has separate weather columns for each region.


---

## Calendar feature creation

- A complete daily calendar table was created using:`pd.date_range("2010-01-01", "2024-12-31", freq="D")`
- This calendar table was used as the merge base to ensure full daily coverage.
- The following calendar features were derived from `date`:`year`,`month`, `day_of_week`, `is_weekend`

Definition of day_of_week: Monday = 0, Tuesday = 1, Wednesday = 2, Thursday = 3, Friday = 4, Saturday = 5, Sunday = 6

---

## Merge logic

A complete daily calendar table was used as the starting base table. The final dataset was built by merging in this order:

### 1. Calendar + demand
Merged on: `date`

Added: `nd_daily_mwh`

### 2. Add holiday flags
Merged on: `date`

Added:`is_holiday_eng_wales`, `is_holiday_scotland`, `is_holiday_gb_any`

### 3. Add reshaped weather table
Merged on: `date`

Added: `tasmin_eng_wales_c`, `tasmax_eng_wales_c`, `tmean_eng_wales_c`, `tasmin_scotland_c`, `tasmax_scotland_c`, `tmean_scotland_c`

### Join type
The joins were done using `how="left"` from the calendar base table.

---

"""

# Save README as a markdown file
readme_path = OBJ2_VALIDATION_DIR / "demand_model_training_README.md"
readme_path.parent.mkdir(parents=True, exist_ok=True)
readme_path.write_text(readme_text, encoding="utf-8")

print(f"Saved file: {readme_path}")